In [2]:
import spacy
from gensim.models import FastText
import pandas as pd
import numpy as np
import re

In [3]:
df = pd.read_csv(r"output_gender_masked.csv")

In [4]:
df = df.dropna(subset=["zwischenfrage_text"])

In [5]:
nlp = spacy.load("de_core_news_lg", disable=["ner"])

def preprocess_parliament_data(texts, batch_size=500):
    """
    Efficiently lemmatizes and cleans a list of German texts.
    """
    cleaned_docs = []
    
    # nlp.pipe is the fastest way to process a large DataFrame
    for doc in nlp.pipe(texts, batch_size=batch_size, n_process=-1):
        # 1. Lemmatization
        # 2. Stop word removal
        # 3. Punctuation/Number removal (is_alpha)
        # 4. Lowercasing
        tokens = [
            token.lemma_.lower() 
            for token in doc 
            if not token.is_stop and not token.is_punct and token.is_alpha
        ]
        cleaned_docs.append(tokens)
        
    return cleaned_docs

# 3. Apply to your DataFrame
# Assuming your text column is named 'question_text'
print("Starting preprocessing...")
df['tokens'] = preprocess_parliament_data(df['zwischenfrage_text'].astype(str).tolist())
print("Finished! Preview:")
print(df['tokens'].head())

Starting preprocessing...
Finished! Preview:
0    [gestatten, herr, kollege, schloß, frage, ausf...
1    [herr, kollege, erler, gestatten, bitte, frage...
2    [wissen, herr, kollege, erler, verstehen, frag...
3    [herr, kollege, erler, glauben, augenblick, ke...
4    [wissen, herr, bundeskanzler, annehmen, berlin...
Name: tokens, dtype: object


In [6]:
# def tokenize_german(text):
#     text = text.lower()
#     tokens = re.findall(r'[a-zäöüß]+', text)
#     return tokens

# tokenized_sentences = [tokenize_german(str(doc)) for doc in df['zwischenfrage_text']]

In [6]:
model = FastText(
    sentences=df['tokens'], 
    vector_size=100, 
    window=8, 
    min_count=10, 
    workers=4,
    sg=1,
    epochs=40  # More epochs for small datasets
)
model.save("parliament_fasttext.model")

In [20]:
model.wv.most_similar(positive=["frauen"], negative=["mann"], topn=30)

[('mißtrauen', 0.4237571954727173),
 ('trauen', 0.41147178411483765),
 ('künstlich', 0.3981296718120575),
 ('vertrauen', 0.39669087529182434),
 ('generell', 0.3409407436847687),
 ('fuß', 0.33156031370162964),
 ('handlungsfähigkeit', 0.3307165801525116),
 ('mitwirkungsrecht', 0.3166559040546417),
 ('verschlechterung', 0.3150653541088104),
 ('mietpreisbremse', 0.30201029777526855),
 ('abbauen', 0.30148130655288696),
 ('tragend', 0.2970205545425415),
 ('josef', 0.29655176401138306),
 ('steuereinnahme', 0.2954930067062378),
 ('handlungsfähig', 0.2889310121536255),
 ('zwischenfrag', 0.28667205572128296),
 ('überkapazität', 0.2866497337818146),
 ('zwischenlager', 0.28017836809158325),
 ('klage', 0.2794033885002136),
 ('stabilitätspolitik', 0.27472054958343506),
 ('kalb', 0.2746124267578125),
 ('realisieren', 0.2735132873058319),
 ('mehreinnahme', 0.27248308062553406),
 ('hierhin', 0.2720474600791931),
 ('gegenüberstehen', 0.2719825506210327),
 ('impuls', 0.268997460603714),
 ('verschlechtern

In [17]:
model.wv.similarity('frau', 'kollegin')

np.float32(0.9043326)

In [56]:
model_three.wv.most_similar(positive=["frau"], negative=["männer"], topn=30)

[('kollegin', 0.46778440475463867),
 ('fuchs', 0.28277912735939026),
 ('ministerin', 0.2808133363723755),
 ('studentin', 0.26465681195259094),
 ('schülerin', 0.2545575201511383),
 ('ehefrau', 0.23641957342624664),
 ('kollegial', 0.2315521389245987),
 ('beförderung', 0.21848028898239136),
 ('umweltministerin', 0.21800242364406586),
 ('hausfrau', 0.21629856526851654),
 ('arbeitsförderungsgesetz', 0.21586096286773682),
 ('hochschule', 0.21243757009506226),
 ('merkel', 0.21109098196029663),
 ('staatssekretärin', 0.20185425877571106),
 ('lehrerin', 0.19980889558792114),
 ('förderung', 0.19184444844722748),
 ('arbeitsförderung', 0.18780937790870667),
 ('höfken', 0.18649636209011078),
 ('präsidentin', 0.18447664380073547),
 ('mädchen', 0.18391495943069458),
 ('erziehen', 0.18356657028198242),
 ('rau', 0.1831020712852478),
 ('verkäuferin', 0.17892152070999146),
 ('süssmuth', 0.17599396407604218),
 ('hierin', 0.17514145374298096),
 ('fachhochschule', 0.17416882514953613),
 ('heiraten', 0.173914

In [7]:
df.source_file = df.source_file.str[3:5]

In [8]:
df.source_file.unique()

<ArrowStringArray>
['02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14',
 '15', '16', '17', '18', '19']
Length: 18, dtype: str

In [9]:
one = ['02', '03', '04', '05']
two = ['06', '07', '08', '09']
three = ['10', '11', '12', '13', '14']
four = ['15', '16', '17', '18', '19']

In [10]:
compass_model = FastText(vector_size=100, window=5, min_count=5, sg=1)
compass_model.build_vocab(df['tokens'])
compass_model.train(df['tokens'], total_examples=len(df), epochs=10)

# 3. Create Decade Models via Fine-tuning
# def get_decade_model(decade_df, base_model):
#     # Copy the base model to preserve the "map" orientation
#     model = FastText(vector_size=100, window=5, min_count=1, sg=1)
#     model.build_vocab_from_freq(base_model.wv.key_to_index)
#     model.wv.vectors[:] = base_model.wv.vectors[:] # Copy weights
    
#     # Fine-tune on just this decade
#     model.train(decade_df['tokens'], total_examples=len(decade_df), epochs=20)
#     return model

def get_decade_model(decade_df, base_model):
    # 1. Initialize with min_count=1 to prevent pruning the base vocab
    model = FastText(vector_size=100, window=5, min_count=1, sg=1)
    
    # 2. Create a dummy frequency dict where every word is above min_count
    # We give every word a frequency of 100 so nothing gets dropped
    full_vocab_dict = {word: 100 for word in base_model.wv.key_to_index.keys()}
    
    # 3. Build the vocab using this dummy dict
    model.build_vocab_from_freq(full_vocab_dict)
    
    # 4. Copy the weights (The shapes will now match perfectly: 9519 to 9519)
    model.wv.vectors[:] = base_model.wv.vectors[:]
    
    # 5. FastText specific: Also copy the n-gram vectors (buckets)
    # This is crucial for FastText to maintain the "German compound" logic
    if hasattr(model.wv, 'vectors_ngrams'):
        model.wv.vectors_ngrams[:] = base_model.wv.vectors_ngrams[:]
    
    # 6. Fine-tune on just this decade
    model.train(decade_df['tokens'], total_examples=len(decade_df), epochs=20)
    
    return model

# Example use:
model_one = get_decade_model(df[df.source_file.isin(one)], compass_model)
model_two = get_decade_model(df[df.source_file.isin(two)], compass_model)
model_three = get_decade_model(df[df.source_file.isin(three)], compass_model)
model_four = get_decade_model(df[df.source_file.isin(four)], compass_model)

In [66]:
np.dot(model_one.wv.get_vector("einkommen"), (model_one.wv.get_mean_vector(["herr", "kollege", "mann"]) - model_one.wv.get_mean_vector(["frau", "kollegin", "dame"])))

np.float32(-0.1519374)

In [24]:
def word_gender_orientation(word, model):
    return np.dot(model.wv.get_vector(word), (model.wv.get_mean_vector(["herr", "kollege", "mann"]) - model.wv.get_mean_vector(["frau", "kollegin", "dame"])))

def word_gender_over_time(word):
    models = {"2-5": model_one, "6-9": model_two, "10-14": model_three, "15-19": model_four}
    for name, model in models.items():
        print(f"In Parlament Generations {name} the word '{word}' has a orientation of {word_gender_orientation(word, model)}")

word_gender_over_time("familie")

In Parlament Generations 2-5 the word 'familie' has a orientation of -0.5213167667388916
In Parlament Generations 6-9 the word 'familie' has a orientation of -0.19825290143489838
In Parlament Generations 10-14 the word 'familie' has a orientation of -0.36182212829589844
In Parlament Generations 15-19 the word 'familie' has a orientation of 0.04057080298662186


In [25]:
model_one.save("parliament_fasttext_2_5.model")
model_two.save("parliament_fasttext_6_9.model")
model_three.save("parliament_fasttext_10_14.model")
model_four.save("parliament_fasttext_15_19.model")

In [27]:
FastText.load("parliament_fasttext_2_5.model")